# 11 — Gate AB-4 analysis: estimand (5%) + applied products (2%), C1–C4, clusters (Python)

Runs AFTER `10_ab4_ensemble` completes all 14 formulations. Kernel `y2y-geo`. Zero solves.
Mirrors the parent's `13_gate4_analysis` (F, E1, E3, E7, E11 at the 5% estimand) and the surface/
tier/cluster half of `19_director_surfaces`, with the Alberta rulings:

- **Two bands.** g = 5% = the estimand (methods mirror: bands, E1 bias, E3 variance shares, E11
  Δ-matrix, C1 like-for-like against the Y2Y-wide F). **g = 2% = the applied band (D-AB10)**: the
  ensemble F₂ *and each value-forward scenario's own f₂* are first-class products (Ethan's
  deliverable definition); tiers = core / per-scenario / opportunity / never.
- **Semantics.** Guarded is the applied headline (mirror); the plain band is carried alongside.
  The guardrails were inert at AB-2 (R5.4) — the side-by-side shows whether that holds across formulations.
- **Level A only** (M9.6). **Cluster constants are AB-scale (D-AB7)** — provisional here
  (`MIN_KM2_AB`, `SIMPLIFY_M`), reported with sensitivity so 12 can set them with disclosure.
- C4 AOI columns use the "alignment, not assignment" reading; tenure split of every tier via the
  03 estimate (over-count disclosure carries).

Outputs → `analysis/ab4/{geotiffs,tables}/`, `analysis/ab4/clusters_2pct.gpkg`, `figures/`, and
`spec/gate_ab4_summary.json`. Every block prints results_log-ready numbers.

**VERSION (AB spec v0.5 D-AB11):** every path comes from `config.ab_paths()` — v3.1 reads `spec/manifest_v3.1.csv` (12 design
formulations on the curated block) and `runs_v3.1/ab_l/A/`, and writes to `analysis/ab4_v3.1/`, `figures/v3.1/`,
`spec/v3.1/`; v1 products stay byte-identical. **The applied band is decided here by rule D-AB13:** the mirror is the parent's
guarded g = 5%; the D-AB10 exception (g = 2%) applies only if the 5% band is FLAT on this block (guarded ensemble frequent
tier < 100 km²). Both bands were solved by 10, so the decision is a read. The E11 objective reconstruction uses each
feature's own target (rarity-scaled EFG targets under v3.1); the crossed diagnostics are not re-solved (the v1 record
stands); M4.31's absolute band widths are tabulated.


In [1]:
# ---- bootstrap ---------------------------------------------------------------------------------------
import hashlib, importlib, json, pathlib, sys, warnings
from datetime import datetime, timezone
from types import SimpleNamespace
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from scipy import ndimage, stats
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)

HERE = ROOT / "analyses" / "alberta_prioritization"
VP = config.ab_paths(); VERSION = VP.version
assert config.EFG_SUBDIR == VP.efg_subdir, f"config.EFG_SUBDIR {config.EFG_SUBDIR!r} does not match VERSION {VERSION}"
SPEC, DATA, REC = HERE / "spec", HERE / "data", VP.records
FIGS, AB4 = VP.figures, VP.analysis; GEO, TAB = AB4 / "geotiffs", AB4 / "tables"
for _d in (FIGS, GEO, TAB, REC):
    _d.mkdir(parents=True, exist_ok=True)
AB = config.AB_HANDOFF_DIR
MAN = pd.read_csv(VP.manifest); assert len(MAN) in (12, 14), len(MAN)
dig = VP.freeze.read_text().split()[0]
assert hashlib.sha256(VP.manifest.read_bytes()).hexdigest() == dig, f"{VP.manifest.name} != freeze hash -- STOP"
LEVEL = MAN.budget_level.unique().tolist(); assert LEVEL == ["A"], LEVEL
RUNS = VP.runs / "A"
HAS_X = bool(MAN.scenario_id.str.endswith("x").any())     # v1 carried the crossed diagnostics; v3.1 does not re-solve them
print(f"VERSION {VERSION}: {VP.manifest.name}; runs {RUNS.relative_to(ROOT)}; EFG block {config.EFG_SUBDIR} ({len(lc.efg_paths(AB))} features)")
FORMS = list(MAN.formulation_id)
SC = json.loads((SPEC / "scenarios_ab_v1.json").read_text()); BLOCKS = SC["_meta"]["blocks"]
EXTENT = json.loads((SPEC / "ab_extent_v1.json").read_text())

# the AB grid as a director_core-compatible object (dc.grid() is hard-wired to the parent stack)
with rasterio.open(AB / "cost_uniform.tif") as src:
    tr, shp, prof = src.transform, src.shape, src.profile
pu = lc.pu_mask(AB)
with rasterio.open(AB / "mask_protected_areas.tif") as src:
    locked2d = (src.read(1) == 1) & pu
G = SimpleNamespace(pu=pu, locked2d=locked2d, locked=locked2d[pu], disc=~locked2d[pu], n_pu=int(pu.sum()),
                    n_disc=int((~locked2d[pu]).sum()), shape=shp, transform=tr, crs=config.TARGET_CRS, profile=prof,
                    cell_km2=abs(tr.a * tr.e) / 1e6)
G.rows, G.cols = np.where(pu)
assert G.n_pu == EXTENT["n_pu"] and G.n_disc == EXTENT["n_unlocked"]
THR, G5, G2 = dc.FREQ_THR, "g05", "g02"
MIN_KM2_AB, SIMPLIFY_M = 10, 500          # D-AB7 provisional AB-scale constants (parent 100 km2 / 2 km); sensitivity below
print(f"{len(FORMS)} formulations at level A | PU {G.n_pu:,} | discretionary {G.n_disc:,} | frequent threshold {THR} | "
      f"second band g={MAN.applied_band_g[0]} (applied band decided in C by rule D-AB13) | cluster min {MIN_KM2_AB} km2 (provisional)")

VERSION v3.1: manifest_v3.1.csv; runs analyses/alberta_prioritization/runs_v3.1/ab_l/A; EFG block iucn_efg_v3 (13 features)
12 formulations at level A | PU 85,133 | discretionary 57,161 | frequent threshold 0.7 | second band g=0.02 (applied band decided in C by rule D-AB13) | cluster min 10 km2 (provisional)


## A — load every formulation once (both bands, both semantics)

In [2]:
# ---- stream: f per (band, semantics), anchors, diameters, unions, certificates --------------------------
TAGS = {"5_plain": "g05", "5_guard": "guard_g05", "2_plain": "g02", "2_guard": "guard_g02"}
F, D, U, ANCH, META = {k: {} for k in TAGS}, {k: {} for k in TAGS}, {k: {} for k in TAGS}, {}, {}
rows = []
for fid in FORMS:
    cd = RUNS / fid
    META[fid] = json.loads((cd / "formulation_meta.json").read_text())
    A = ec.read_selections(cd / "anchor.tif", G.pu)[0]
    ANCH[fid] = A
    m_disc = int(A[G.disc].sum())
    r = dict(formulation=fid, anchor_z=META[fid]["anchor_objective"], drift=META[fid].get("anchor_rel_drift", np.nan), m_disc=m_disc)
    for key, tag in TAGS.items():
        cert = pd.read_csv(cd / f"certificates_{tag}.csv")
        assert bool(cert.band_ok.all()), f"{fid}/{tag}: band certificate violated"
        M = ec.read_selections(cd / f"mga_{tag}.tif", G.pu)
        S = np.vstack([A[None, :], M])
        F[key][fid] = S.mean(axis=0).astype(np.float32)
        D[key][fid] = dc._diam(S, G.disc, m_disc)
        U[key][fid] = S.any(axis=0)
        r[f"D_{key}"] = D[key][fid]; r[f"freq_km2_{key}"] = int(((F[key][fid] >= THR) & G.disc).sum())
        r[f"dup_{key}"] = int(cert.duplicate.sum()); r[f"min_{key}"] = float(cert.runtime_s.sum() / 60)
        del S, M
    rows.append(r)
LOAD = pd.DataFrame(rows).set_index("formulation")
print(LOAD[["anchor_z", "drift", "m_disc", "D_5_plain", "D_5_guard", "D_2_plain", "D_2_guard",
            "freq_km2_5_guard", "freq_km2_2_plain", "freq_km2_2_guard"]].to_string(float_format=lambda v: f"{v:.4g}"))
print(f"\nsweep time total {LOAD[[c for c in LOAD if c.startswith('min_')]].sum().sum()/60:.1f} h | "
      f"duplicates {int(LOAD[[c for c in LOAD if c.startswith('dup_')]].sum().sum())}")
LOAD.to_csv(TAB / "load_summary.csv")

                  anchor_z     drift  m_disc  D_5_plain  D_5_guard  D_2_plain  D_2_guard  freq_km2_5_guard  freq_km2_2_plain  freq_km2_2_guard
formulation                                                                                                                                   
s0_ssp585_theta5     4.061 4.729e-06   10083          1          1          1          1                56                 2                41
s1_ssp585_theta5     3.902 5.911e-06   10083          1          1     0.9765     0.9767                 8               206               206
s2_ssp585_theta5     4.053 2.191e-07   10083          1          1     0.9973     0.9826               201                39               209
s3_ssp585_theta5     4.245 7.594e-06   10083          1          1          1          1                14                 0                14
s4_ssp585_theta2     3.884   3.4e-06   10083          1          1     0.9999     0.9998                26                 5                12

## B — the estimand (g = 5%): F, bands, E1, E3, E11 — the methods mirror

In [3]:
# ---- F (hierarchical, one vote per formulation), bands, E1 bias, E2 note --------------------------------
F5p, F5g = dc.ensemble(F["5_plain"], FORMS), dc.ensemble(F["5_guard"], FORMS)
Fn = np.mean([ANCH[c] for c in FORMS], axis=0).astype(np.float32)          # E1 naive: anchors only
B5 = dc.band_table(G, {"plain": F5p, "guarded": F5g}); B5.to_csv(TAB / "bands_5pct.csv", index=False)
print("F bands at g=5% (discretionary km2):"); print(B5.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
bias = F5p - Fn
print(f"\nE1 bias (F_hier - F_naive): mean |bias| {np.abs(bias[G.disc]).mean():.4f} | max {np.abs(bias[G.disc]).max():.3f} | "
      f"cells |bias|>0.1: {int((np.abs(bias) > 0.1)[G.disc].sum()):,}")
print("E2: equal k across formulations -> flat pool == hierarchical mean (definitional)")
for name, v in (("F5_plain", F5p), ("F5_guard", F5g), ("E1_bias", bias), ("F5_naive_anchors", Fn)):
    dc.write_tif(G, v, GEO / f"{name}.tif")
# E3 variance decomposition on the 12 factorial formulations + crossed contrast
fact = [c for c in FORMS if not c.startswith(("s1x", "s3x"))]
Ff = np.stack([F["5_plain"][c] for c in fact])
V_between, V_within = Ff.var(axis=0), np.mean([F["5_plain"][c] * (1 - F["5_plain"][c]) for c in FORMS], axis=0)
scen = np.array([c.split("_")[0] for c in fact]); clim = np.array([c.split("_")[1] for c in fact])
V_scen = np.stack([Ff[scen == s].mean(0) for s in np.unique(scen)]).var(0)
V_clim = np.stack([Ff[clim == k].mean(0) for k in np.unique(clim)]).var(0)
tot = V_between + V_within + 1e-12
E3 = dict(within=float((V_within/tot)[G.disc].mean()), between=float((V_between/tot)[G.disc].mean()),
          scenario=float((V_scen/tot)[G.disc].mean()), climate=float((V_clim/tot)[G.disc].mean()))
XREG = ({p: float(np.abs(F["5_plain"][[c for c in FORMS if c.startswith(p + "x")][0]] - F["5_plain"][f"{p}_ssp585_theta5"])[G.disc].mean()) for p in ("s1", "s3")}
        if HAS_X else "crossed diagnostics not re-solved under this VERSION -- the v1 record (R7.1) stands as evidence (parent v0.17)")
print(f"\nE3 variance shares (disc means): within {E3['within']:.3f} | between {E3['between']:.3f} "
      f"(scenario {E3['scenario']:.3f}, climate {E3['climate']:.3f}) [parent 0.952 / 0.044 / 0.002] | crossed contrast mean|Δf| {XREG}")
print(f"D_s at 5% (plain): min {min(D['5_plain'].values()):.3f} / max {max(D['5_plain'].values()):.3f} | guarded: "
      f"min {min(D['5_guard'].values()):.3f} / max {max(D['5_guard'].values()):.3f}")
# M4.31 (mirror): the band is RELATIVE to z*, so a formulation change that removes inert shortfall tightens the absolute tolerance
V1R = config.ab_paths("v1").runs / "A"
rows_ = []
for fid in FORMS:
    z = META[fid]["anchor_objective"]; r_ = dict(formulation=fid, z_star=z, width_abs_g05=0.05 * z, width_abs_g02=0.02 * z)
    p1 = V1R / fid / "formulation_meta.json"
    if VERSION != "v1" and p1.exists():
        z1 = json.loads(p1.read_text())["anchor_objective"]; r_.update(z_star_v1=z1, z_change_pct=100 * (z - z1) / z1)
    rows_.append(r_)
BW = pd.DataFrame(rows_).set_index("formulation"); BW.to_csv(TAB / "band_width_abs.csv")
print("\nM4.31 disclosure -- z* and the ABSOLUTE band widths (5% and 2% of z*)" + (", vs the v1 block" if "z_star_v1" in BW.columns else "") + ":")
print(BW.to_string(float_format=lambda v: f"{v:.4f}"))


F bands at g=5% (discretionary km2):
                    band  plain km2  plain %disc  guarded km2  guarded %disc
      never [0.00, 0.05)          0          0.0            0            0.0
       rare [0.05, 0.30)      57161        100.0        53833           94.2
conditional [0.30, 0.70)          0          0.0         3301            5.8
   frequent [0.70, 0.95)          0          0.0           27            0.0
     always [0.95, 1.00]          0          0.0            0            0.0

E1 bias (F_hier - F_naive): mean |bias| 0.2405 | max 0.825 | cells |bias|>0.1: 50,137
E2: equal k across formulations -> flat pool == hierarchical mean (definitional)

E3 variance shares (disc means): within 1.000 | between 0.000 (scenario 0.000, climate 0.000) [parent 0.952 / 0.044 / 0.002] | crossed contrast mean|Δf| crossed diagnostics not re-solved under this VERSION -- the v1 record (R7.1) stands as evidence (parent v0.17)
D_s at 5% (plain): min 1.000 / max 1.000 | guarded: min 0.999 / max 

In [4]:
# ---- E7 (T1): anchor captures + theta-tail rates; E11: Jaccard + Delta(s,s') with diagonal self-check -----
theta = config.AUDIT["theta"]; cont = lc.continuous_features()
A_mat = np.stack([ANCH[c] for c in FORMS])
vals = {f: np.nan_to_num(lc._read(AB / f"{f}.tif")[G.pu], nan=0.0) for f in cont}
caps = {f: (A_mat @ v) / v.sum() for f, v in vals.items()}
tails = {}
for f, v in vals.items():
    tl = v >= theta * v.mean()
    tails[f] = (A_mat[:, tl] @ v[tl]) / v[tl].sum() if tl.any() else np.full(len(FORMS), np.nan)
efg_paths = lc.efg_paths(AB); n_efg = len(efg_paths)
efg_caps = {p.stem: (A_mat @ np.nan_to_num(lc._read(p)[G.pu], nan=0.0)) / max(np.nan_to_num(lc._read(p)[G.pu], nan=0.0).sum(), 1e-12) for p in efg_paths}
T1 = pd.DataFrame(caps, index=FORMS); T1["EFG_mean"] = pd.DataFrame(efg_caps, index=FORMS).mean(axis=1)
TT = pd.DataFrame(tails, index=FORMS)
banked = {f: float(v[G.locked].sum() / v.sum()) for f, v in vals.items()}
print("anchor captures per formulation (banked share in parentheses):")
print(T1.rename(columns=lambda c: c.replace("irrecoverable_carbon_", "").replace("climate_type_", "").replace("transboundary_", "").replace("aoh_richness_", "")).round(3).to_string())
print("banked:", {k.replace("irrecoverable_carbon_", "").replace("climate_type_", "")[:14]: round(v, 3) for k, v in banked.items()})
print("\ntheta-tail mass capture (m_soc, biomass, macrorefugia, connectivity):")
print(TT[["irrecoverable_carbon_m_soc", "irrecoverable_carbon_biomass", "climate_type_macrorefugia", "transboundary_connectivity"]].round(3).to_string())
T1.to_csv(TAB / "T1_anchor_captures.csv"); TT.to_csv(TAB / "T1_tail_capture.csv")
ghm = vals["human_modification"]
for c in [x for x in FORMS if x.startswith("s3")]:
    new = ANCH[c] & G.disc
    print(f"gHM audit {c}: mean raw gHM new-selection {float((1-ghm)[new].mean()):.4f} vs unselected {float((1-ghm)[~ANCH[c] & G.disc].mean()):.4f}")

# E11
J = pd.DataFrame(index=FORMS, columns=FORMS, dtype=float)
for a in FORMS:
    for b in FORMS:
        Sa, Sb = ANCH[a][G.disc], ANCH[b][G.disc]
        J.loc[a, b] = float((Sa & Sb).sum() / max((Sa | Sb).sum(), 1))
off = J.values[~np.eye(len(FORMS), dtype=bool)]
print(f"\nE11 between-anchor discretionary Jaccard: min {off.min():.3f} / mean {off.mean():.3f} / max {off.max():.3f} [parent 0.373-0.931]")
v245 = np.nan_to_num(lc._read(AB / "climate_realizations" / "macrorefugia_245_2071_2100.tif")[G.pu], nan=0.0)
caps245 = (A_mat @ v245) / v245.sum()
def objective_of(i, form):
    row = MAN.set_index("formulation_id").loc[form]; w, t = json.loads(row.weight_vector), json.loads(row.target_vector)
    obj = 0.0
    for f in cont:
        wf, tf = float(w.get(f, 1.0)), float(t.get(f, 1.0))
        cap = caps245[i] if (f == "climate_type_macrorefugia" and "ssp245" in form) else caps[f][i]
        obj += wf * max(0.0, tf - cap) / tf
    for e, cv in efg_caps.items():                      # per-EFG targets (v3.1: rarity-scaled, by feature name; v1: 1.0)
        te = float(t.get(e, 1.0))
        obj += (1.0 / n_efg) * max(0.0, te - cv[i]) / te
    return obj
DM = pd.DataFrame(index=FORMS, columns=FORMS, dtype=float)
for i, a in enumerate(FORMS):
    for b in FORMS:
        zb = META[b]["anchor_objective"]; DM.loc[a, b] = (objective_of(i, b) - zb) / zb
diag = np.diag(DM.values.astype(float))
print(f"Delta diagonal self-check: max |diag| {np.abs(diag).max():.2e} (expect ~0)")
inband = (DM.values.astype(float) <= 0.05 + 1e-9)
n_pairs = len(FORMS) ** 2 - len(FORMS)
E11 = dict(pairs_in_band=int(inband.sum() - len(FORMS)), pairs=n_pairs, diag_max=float(np.abs(diag).max()),
           J_min=float(off.min()), J_mean=float(off.mean()), J_max=float(off.max()))
print(f"anchors inside each other's 5% bands: {E11['pairs_in_band']}/{n_pairs} ordered pairs [parent 156/182]")
worst = DM.astype(float).where(~np.eye(len(FORMS), dtype=bool)).stack().nlargest(5)
print("largest cross-objective suboptimalities:\n" + worst.to_string(float_format=lambda v: f"{v:.4f}"))
J.to_csv(TAB / "E11_anchor_jaccard.csv"); DM.to_csv(TAB / "E11_delta_matrix.csv")

anchor captures per formulation (banked share in parentheses):
                  human_modification  connectivity  climate_corridors  macrorefugia  biomass  m_soc  mammals  birds  EFG_mean
s0_ssp585_theta5               0.481         0.574              0.478         0.586    0.418  0.743    0.450  0.428     0.549
s1_ssp585_theta5               0.481         0.561              0.476         0.597    0.393  0.770    0.448  0.426     0.552
s2_ssp585_theta5               0.480         0.587              0.483         0.576    0.405  0.752    0.448  0.425     0.549
s3_ssp585_theta5               0.481         0.561              0.477         0.580    0.416  0.732    0.452  0.429     0.548
s4_ssp585_theta2               0.481         0.564              0.477         0.578    0.459  0.772    0.449  0.426     0.550
s5_ssp585_theta5               0.483         0.572              0.478         0.585    0.413  0.752    0.449  0.427     0.550
s0_ssp245_theta5               0.481         0.574     

## C — the applied band (rule D-AB13: the parent's guarded 5% unless the 5% band is flat on this block, then 2% per D-AB10): ensemble F, per-scenario f, tiers


In [5]:
# ---- D-AB13 (AB spec v0.5): the applied band. Mirror = the parent's guarded g = 5%; the D-AB10 exception (g = 2%) applies
# only if the 5% band is FLAT on this block: guarded ensemble frequent tier (F >= 0.70) below FLAT_KM2 (1% of the additions
# budget). Pre-stated; both bands were solved by 10, so the decision is a read, not a re-solve.
FLAT_KM2 = 100
freq5 = int(((F5g >= THR) & G.disc).sum())
APPLIED = "2" if freq5 < FLAT_KM2 else "5"
print(f"D-AB13: guarded frequent tier at 5% = {freq5:,} km2 (flat if < {FLAT_KM2}) -> APPLIED BAND g = {APPLIED}%"
      + (" (the D-AB10 exception)" if APPLIED == "2" else " (parent mirror)"))
FA = {sem: F[f"{APPLIED}_{sem}"] for sem in ("plain", "guard")}; UA = U[f"{APPLIED}_guard"]
# ---- F at the applied band (both semantics), per-scenario surfaces, climate pooling, tiers ---------------------------------
F2p, F2g = dc.ensemble(FA["plain"], FORMS), dc.ensemble(FA["guard"], FORMS)          # F2* = the APPLIED band (name kept from v1)
U2g = np.mean([UA[c] for c in FORMS], axis=0).astype(np.float32)
B2 = dc.band_table(G, {"plain": F2p, "guarded": F2g}); B2.to_csv(TAB / "bands_applied.csv", index=False)
print(f"F bands at the applied g={APPLIED}% (discretionary km2):"); print(B2.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
tierJ = dc.jaccard((F2p >= THR) & G.disc, (F2g >= THR) & G.disc)
print(f"guarded vs plain frequent tier at {APPLIED}%: Jaccard {tierJ:.3f} (guardrails {'inert' if tierJ > 0.95 else 'ACTIVE'})")
for name, v in (("F_applied_plain", F2p), ("F_applied_guard", F2g), ("union_applied_guard", U2g)):
    dc.write_tif(G, v, GEO / f"{name}.tif")
for fid in FORMS:
    dc.write_tif(G, FA["guard"][fid], GEO / f"f_applied_guard_{fid}.tif")

POOL, prep = dc.pool_scenarios(G, FA["guard"], MAN)      # decision (g) mirror: pool climate levels unless tiers diverge
prep.to_csv(TAB / "pooling_check_applied.csv", index=False)
print("\nclimate pooling (frequent-tier Jaccard between levels; pool iff >= 0.80):")
print(prep.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
for key, f in POOL.items():
    dc.write_tif(G, f, GEO / f"f_applied_guard_pooled_{key.replace('@', '_')}.tif")

core = (F2g >= THR) & G.disc
tiers = {"core (F >= 0.70, all formulations)": core}
any_sc = np.zeros(G.n_pu, bool)
for key, f in POOL.items():
    sid = key.split("@")[0]
    if sid in ("s0", "s5") or sid.endswith("x"):        # value-forward = S1..S4; S0/S5/crossed to the appendix
        continue
    t = (f >= THR) & G.disc & ~core
    tiers[f"{key}: {dc.SCENARIO_LABEL.get(sid, sid)} (frequent minus core)"] = t
    any_sc |= t
tiers["any value-forward scenario (union)"] = any_sc
tiers["opportunity (in >= 1 plan of >= 1 formulation, not above)"] = (U2g > 0) & G.disc & ~core & ~any_sc
tiers[f"never (no {APPLIED}%-band plan selects it)"] = (U2g == 0) & G.disc
ACT = pd.DataFrame([{"tier": k, "km2": int(m.sum()), "pct_disc": 100 * m.sum() / G.n_disc} for k, m in tiers.items()])
ACT.to_csv(TAB / "tiers_applied.csv", index=False)
print(f"\ntiers at the applied g={APPLIED}% (guarded):"); print(ACT.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
tier_code = np.zeros(G.n_pu, np.uint8)
tier_code[tiers["opportunity (in >= 1 plan of >= 1 formulation, not above)"]] = 1
tier_code[any_sc] = 2; tier_code[core] = 3
dc.write_tif(G, tier_code, GEO / "tiers_applied.tif", dtype="uint8", nodata=255)

D-AB13: guarded frequent tier at 5% = 27 km2 (flat if < 100) -> APPLIED BAND g = 2% (the D-AB10 exception)
F bands at the applied g=2% (discretionary km2):
                    band  plain km2  plain %disc  guarded km2  guarded %disc
      never [0.00, 0.05)       2110          3.7         2020            3.5
       rare [0.05, 0.30)      51226         89.6        49978           87.4
conditional [0.30, 0.70)       3825          6.7         5132            9.0
   frequent [0.70, 0.95)          0          0.0           28            0.0
     always [0.95, 1.00]          0          0.0            3            0.0
guarded vs plain frequent tier at 2%: Jaccard 0.000 (guardrails ACTIVE)

climate pooling (frequent-tier Jaccard between levels; pool iff >= 0.80):
scenario  levels  jaccard            decision  freq_km2_585  freq_km2_245
      s0       2    0.818              POOLED            41            39
      s1       2    0.193 SEPARATE (diverged)           206           701
      s2     

PosixPath('/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/analyses/alberta_prioritization/analysis/ab4_v3.1/geotiffs/tiers_applied.tif')

## D — C1: do Alberta's priorities concur with the Alberta clip of the Y2Y-wide priorities?

Like-for-like only: the parent's 5% hierarchical F (plain) vs AB's F₅ (plain), on AB discretionary
cells; anchors compared per formulation by the **overlap coefficient** (Jaccard supplementary —
the clip's selected share ≠ AB's additions share). At 2% only the reference formulation exists on
the parent side.

In [6]:
# ---- C1 ------------------------------------------------------------------------------------------------
YP = config.y2y_paths(VERSION); Y2Y_RUNS = YP.runs                       # the parent at the SAME version = block-matched (D-AB11)
with rasterio.open(Y2Y_RUNS / ("ensemble_v1" if VERSION == "v1" else "ensemble") / "F_surface.tif") as s:
    Fy = np.nan_to_num(s.read(1), nan=0.0)[G.pu]
rho = stats.spearmanr(Fy[G.disc], F5p[G.disc]).correlation
tA, tY = (F5p >= THR) & G.disc, (Fy >= THR) & G.disc
ovl = lambda a, b: float((a & b).sum() / max(min(a.sum(), b.sum()), 1))
print(f"C1 (5%, plain): Spearman(F_AB, F_Y2Y|AB) over {G.n_disc:,} discretionary cells = {rho:.3f} | frequent tiers AB {tA.sum():,} / "
      f"Y2Y-clip {tY.sum():,} km2 -> overlap coeff {ovl(tA, tY):.3f}, Jaccard {dc.jaccard(tA, tY):.3f}")
print(f"   Y2Y-wide F inside AB: mean over discretionary {Fy[G.disc].mean():.3f} (AB F5 {F5p[G.disc].mean():.3f}); "
      f"Y2Y-clip cells with F>=0.05: {int(((Fy >= 0.05) & G.disc).sum()):,}")
rows = []
for fid in FORMS:
    pfid = fid.replace("theta2", "theta3")            # the parent's S4/crossed regime label
    pa = Y2Y_RUNS / pfid / "anchor.tif"
    if not pa.exists(): continue
    Ay = ec.read_selections(pa, G.pu)[0] & G.disc; Aa = ANCH[fid] & G.disc
    rows.append(dict(formulation=fid, parent=pfid, ab_additions=int(Aa.sum()), y2y_clip_selected=int(Ay.sum()),
                     overlap_coeff=round(ovl(Aa, Ay), 4), jaccard=round(dc.jaccard(Aa, Ay), 4)))
C1 = pd.DataFrame(rows); C1.to_csv(TAB / "C1_anchor_concordance.csv", index=False)
print("\nC1 anchors (AB additions vs the Y2Y-wide anchor's selection inside AB, discretionary cells):")
print(C1.to_string(index=False))
# 2% on the reference only (a v1 probe: the parent never solved a 2% band on the curated block)
p2 = config.y2y_paths("v1").runs / "ensemble_v1" / "mga_g02.tif" if VERSION == "v1" else None
if p2 is not None and p2.exists():
    Sy = np.vstack([ec.read_selections(Y2Y_RUNS / "ensemble_v1" / "anchor.tif", G.pu), ec.read_selections(p2, G.pu)])
    fy2 = Sy.mean(0); fa2 = F["2_plain"]["s0_ssp585_theta5"]
    rho2 = stats.spearmanr(fy2[G.disc], fa2[G.disc]).correlation
    print(f"\nC1 (2%, S0 reference): Spearman {rho2:.3f} | tiers AB {int(((fa2 >= THR) & G.disc).sum()):,} / Y2Y-clip "
          f"{int(((fy2 >= THR) & G.disc).sum()):,} km2 -> overlap {ovl((fa2 >= THR) & G.disc, (fy2 >= THR) & G.disc):.3f}")
else:
    print("\nC1 (2%, S0 reference): not available on this VERSION (v1 probe only)")
C1S = dict(spearman_5pct=float(rho), tier_overlap_5pct=ovl(tA, tY), tier_jaccard_5pct=dc.jaccard(tA, tY),
           anchors=C1.to_dict(orient="records"), parent_version=VERSION)
# C1b (applied, guarded): the parent's guarded F (its director package at the SAME version) vs Alberta's applied guarded F
PKGY = ROOT / "analyses" / "y2y" / "director_package"
if (PKGY / "summary.json").exists() and json.loads((PKGY / "summary.json").read_text()).get("version", "v1") == VERSION:
    with rasterio.open(PKGY / "geotiffs" / "F_guarded.tif") as s:
        Fyg = np.nan_to_num(s.read(1), nan=0.0)[G.pu]
    rhog = stats.spearmanr(Fyg[G.disc], F2g[G.disc]).correlation
    tAg, tYg = (F2g >= THR) & G.disc, (Fyg >= THR) & G.disc
    print(f"\nC1b (guarded; parent 5% vs Alberta applied {APPLIED}%): Spearman {rhog:.3f} | core AB {tAg.sum():,} / Y2Y-clip {tYg.sum():,} km2 -> "
          f"overlap {ovl(tAg, tYg):.3f}, Jaccard {dc.jaccard(tAg, tYg):.3f}")
    C1S["guarded"] = dict(spearman=float(rhog), overlap=ovl(tAg, tYg), jaccard=dc.jaccard(tAg, tYg), ab_core_km2=int(tAg.sum()), y2y_clip_core_km2=int(tYg.sum()))
else:
    C1S["guarded"] = "parent director package not at this VERSION (run the parent's 19 first)"; print("\nC1b skipped:", C1S["guarded"])

C1 (5%, plain): Spearman(F_AB, F_Y2Y|AB) over 57,161 discretionary cells = 0.284 | frequent tiers AB 0 / Y2Y-clip 0 km2 -> overlap coeff 0.000, Jaccard nan
   Y2Y-wide F inside AB: mean over discretionary 0.177 (AB F5 0.176); Y2Y-clip cells with F>=0.05: 57,157

C1 anchors (AB additions vs the Y2Y-wide anchor's selection inside AB, discretionary cells):
     formulation           parent  ab_additions  y2y_clip_selected  overlap_coeff  jaccard
s0_ssp585_theta5 s0_ssp585_theta5         10083              13373         0.8600   0.5865
s1_ssp585_theta5 s1_ssp585_theta5         10083              15851         0.9302   0.5665
s2_ssp585_theta5 s2_ssp585_theta5         10083               7429         0.8542   0.5683
s3_ssp585_theta5 s3_ssp585_theta5         10083              10497         0.7299   0.5567
s4_ssp585_theta2 s4_ssp585_theta3         10083               9066         0.7459   0.5459
s5_ssp585_theta5 s5_ssp585_theta5         10083              11968         0.8313   0.6132
s0_ssp2

## E — C4 AOI overlays + tenure split of the tiers + distance-to-PA reading

In [7]:
# ---- C4 + tenure + distance --------------------------------------------------------------------------------
def rd(p):
    with rasterio.open(p) as s: return s.read(1)
nfz = (rd(DATA / "derived" / "aoi_nfz_novel.tif") == 1)[G.pu]
ten = rd(DATA / "derived" / "tenure_class.tif")[G.pu]
dist = rd(DATA / "derived" / "dist_to_pa_km.tif")[G.pu]
srp = gpd.read_file("/vsizip/" + str(DATA / "aoi" / "US_SRP_PlanningBoundary.zip") + "/Data/US_SRP_PlanningBoundary.shp").to_crs(G.crs)
srp1 = rasterize([(g, 1) for g in srp.geometry], out_shape=G.shape, transform=G.transform, fill=0, dtype="uint8").astype(bool)[G.pu] & G.disc
AOIS = {"AOI-2 Nature First (novel)": nfz & G.disc, "Upper Smoky SRP planning area (unlocked)": srp1}
rows = []
for name, m in AOIS.items():
    null = 100 * m.sum() / G.n_disc
    r = dict(aoi=name, km2=int(m.sum()), share_of_disc_pct=null, mean_F2_inside=float(F2g[m].mean()), mean_F2_disc=float(F2g[G.disc].mean()),
             core_inside_km2=int((core & m).sum()), core_inside_pct=100 * (core & m).sum() / max(core.sum(), 1),
             ratio_vs_null=(100 * (core & m).sum() / max(core.sum(), 1)) / null if null else np.nan)
    for key, f in POOL.items():
        r[f"mean_f2_{key}"] = float(f[m].mean())
    rows.append(r)
C4 = pd.DataFrame(rows); C4.to_csv(TAB / "C4_aoi.csv", index=False)
print("C4 (alignment, not assignment): AOI share of discretionary land = the null; core inside = capture):")
print(C4[["aoi", "km2", "share_of_disc_pct", "core_inside_km2", "core_inside_pct", "ratio_vs_null", "mean_F2_inside", "mean_F2_disc"]].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

NAMES = {2: "crown_green", 3: "crown_white_ind", 4: "private_presumed", 5: "private_ranchland", 6: "unclassified"}
rows = []
for k, m in tiers.items():
    r = {"tier": k, "km2": int(m.sum())}
    for code_, nm in NAMES.items():
        r[nm] = int((m & (ten == code_)).sum())
    rows.append(r)
TEN = pd.DataFrame(rows); TEN.to_csv(TAB / "tenure_split_applied.csv", index=False)
print(f"\ntenure split of the applied {APPLIED}% tiers (km2; private classes are OVER-counts by the crown-lease share, D-AB8):")
print(TEN.to_string(index=False))
print(f"   OECM-track share of the core: {100 * TEN.iloc[0].private_ranchland / max(TEN.iloc[0].km2, 1):.1f}% | protection-track "
      f"{100 * (TEN.iloc[0].crown_green + TEN.iloc[0].crown_white_ind) / max(TEN.iloc[0].km2, 1):.1f}%")
bands_km = [(0, 5), (5, 10), (10, 20), (20, 1e9)]
print("\ndistance to nearest PA (the tabled D-AB6 reading): core vs discretionary null")
for lo, hi in bands_km:
    mm = (dist > lo) & (dist <= hi)
    print(f"   {lo:>3}-{'inf' if hi > 1e8 else int(hi):<4} km: core {100 * (core & mm).sum() / max(core.sum(), 1):5.1f}% | null {100 * (mm & G.disc).sum() / G.n_disc:5.1f}%")

C4 (alignment, not assignment): AOI share of discretionary land = the null; core inside = capture):
                                     aoi   km2  share_of_disc_pct  core_inside_km2  core_inside_pct  ratio_vs_null  mean_F2_inside  mean_F2_disc
              AOI-2 Nature First (novel)   436               0.76                0             0.00           0.00            0.34          0.18
Upper Smoky SRP planning area (unlocked) 10028              17.54                0             0.00           0.00            0.19          0.18

tenure split of the applied 2% tiers (km2; private classes are OVER-counts by the crown-lease share, D-AB8):
                                                     tier   km2  crown_green  crown_white_ind  private_presumed  private_ranchland  unclassified
                       core (F >= 0.70, all formulations)    31           29                0                 0                  0             2
       s1@585: Core-habitat-forward (frequent minus core)   183  

## F — clusters at the applied band (pre-stated procedure; AB-scale constants provisional, D-AB7)

In [8]:
# ---- clusters: core + per-scenario (minus core); register with AB-local star profiles ---------------------------
core2d = dc.to_grid(G, core, fill=False, dtype=bool)
lab1, reg1 = dc.clusters(G, F2g, thr=THR, min_km2=MIN_KM2_AB, close_r=dc.CLOSE_R)
reg1.insert(0, "act", "core"); reg1.insert(1, "key", "ensemble")
print(f"core: tier {int(core.sum()):,} km2 -> {len(reg1)} components, {int(reg1.kept.sum())} >= {MIN_KM2_AB} km2 ({reg1[reg1.kept].km2.sum():,.0f} km2)")
sens = [dc.sensitivity(G, F2g, min_km2=MIN_KM2_AB).assign(act="core", key="ensemble")]
for mk in (5, 10, 25, 50):
    _, rr = dc.clusters(G, F2g, thr=THR, min_km2=mk); print(f"   min size {mk:>2} km2: {int(rr.kept.sum())} clusters, {rr[rr.kept].km2.sum():,.0f} km2 kept")
LABELS, REGS = {"core": lab1}, [reg1]
for key, f in POOL.items():
    sid = key.split("@")[0]
    if sid in ("s0", "s5") or sid.endswith("x"): continue
    lab, reg = dc.clusters(G, f, thr=THR, min_km2=MIN_KM2_AB, close_r=dc.CLOSE_R, subtract2d=core2d)
    reg.insert(0, "act", "scenario"); reg.insert(1, "key", key)
    LABELS[key] = lab; REGS.append(reg)
    sens.append(dc.sensitivity(G, f, min_km2=MIN_KM2_AB, subtract2d=core2d).assign(act="scenario", key=key))
    kept = reg[reg.kept]
    print(f"scenario {key:<8} ({dc.SCENARIO_LABEL.get(sid, sid)}): {len(reg)} components, {len(kept)} kept after core subtraction "
          f"({kept.residual_km2.sum() if len(kept) else 0:,.0f} km2 residual)")
REG = pd.concat(REGS, ignore_index=True); pd.concat(sens, ignore_index=True).to_csv(TAB / "cluster_sensitivity.csv", index=False)

# AB-local block percentiles (over AB discretionary land) + representativeness = EFG classes present / n_efg
axes = {}
pct = {}
for f in sorted({x for d in dc.BLOCK_AXES.values() for x in d}):
    v = vals[f]; ref = np.sort(v[G.disc]); pct[f] = (np.searchsorted(ref, v, side="right") / len(ref)).astype(np.float32)
for ax, members in dc.BLOCK_AXES.items():
    axes[ax] = sum(w * pct[f] for f, w in members.items()).astype(np.float32)
efg = np.stack([np.nan_to_num(lc._read(p)[G.pu], nan=0.0) > 0 for p in efg_paths])
soc = vals["irrecoverable_carbon_m_soc"]; soc_tail = soc >= theta * soc.mean()
WINF = REC / "efg_window_footprints.csv"                               # v3.1: rarity judged in the Alberta +250 km window (D-AB12)
if WINF.exists():
    _w = pd.read_csv(WINF).set_index("feature"); _rare = set(_w.index[_w.rare_window])
    rare_sel = np.array([p.stem in _rare for p in efg_paths]); rare_rule = f"rare in the extent+{config.EFG_TARGET_WINDOW_KM} km window"
else:
    rare_sel = efg.mean(axis=1) <= dc.RARE_EFG_PCT; rare_rule = "<= 1% of the AB PU (v1 rule)"       # EFGs present on <= 1% of the AB PU
rare_efg = efg[rare_sel].any(axis=0) if rare_sel.any() else np.zeros(G.n_pu, bool)
conn = vals["transboundary_connectivity"]; conn_spike = conn >= np.quantile(conn[G.disc], 0.99)
print(f"driver masks: m_soc theta-tail {int(soc_tail[G.disc].sum()):,} disc cells | rare EFGs ({rare_rule}) {int(rare_sel.sum())} classes "
      f"covering {int(rare_efg[G.disc].sum()):,} | connectivity spike (top 1%) {int(conn_spike[G.disc].sum()):,}")
near_pa = dist <= 5
rows = []
for _, r in REG[REG.kept].iterrows():
    lab = LABELS["core" if r.act == "core" else r.key]
    m2 = (lab == r.cid)
    if r.act == "scenario": m2 = m2 & ~core2d
    m1 = m2[G.pu]
    prof_ = {ax: float(axes[ax][m1].mean()) for ax in dc.BLOCK_AXES}
    prof_["representativeness"] = float(efg[:, m1].any(axis=1).sum() / efg.shape[0])
    row = dict(act=r.act, key=r.key, cid=int(r.cid), km2=float(m1.sum() * G.cell_km2), mean_F2=float(F2g[m1].mean()),
               min_F2=float(F2g[m1].min()), lat=float(r.lat), lon=float(r.lon))
    row.update({f"pct_{k}": v for k, v in prof_.items()})
    row.update(driver_msoc_tail=100 * float(soc_tail[m1].mean()), driver_rare_efg=100 * float(rare_efg[m1].mean()),
               driver_conn_spike=100 * float(conn_spike[m1].mean()), pct_within_5km_PA=100 * float(near_pa[m1].mean()),
               pct_in_NFZ=100 * float(nfz[m1].mean()), pct_private_ranchland=100 * float((ten[m1] == 5).mean()),
               pct_crown=100 * float(np.isin(ten[m1], [2, 3]).mean()),
               n_formulations_frequent=int(sum(float(F["2_guard"][fid][m1].mean() >= THR) for fid in FORMS)))
    rows.append(row)
TD1 = pd.DataFrame(rows).sort_values(["act", "km2"], ascending=[True, False]).reset_index(drop=True)
TD1.to_csv(TAB / "cluster_register_applied.csv", index=False)
print("\ncluster register (top rows):")
print(TD1[["act", "key", "km2", "mean_F2", "lat", "driver_msoc_tail", "driver_rare_efg", "pct_private_ranchland", "pct_crown", "pct_within_5km_PA", "pct_in_NFZ", "n_formulations_frequent"]].head(20).to_string(index=False, float_format=lambda v: f"{v:.1f}"))
parts = []
for k, lab in LABELS.items():
    ids = REG[(REG.kept) & ((REG.key == "ensemble") if k == "core" else (REG.key == k))].cid.tolist()
    if ids:
        g = dc.vectorize(G, lab if k == "core" else np.where(core2d, 0, lab), ids, simplify_m=SIMPLIFY_M); g["act"] = "core" if k == "core" else "scenario"; g["key"] = k
        parts.append(g)
if parts:
    pd.concat(parts, ignore_index=True).to_file(AB4 / "clusters_applied.gpkg", driver="GPKG")
    print(f"wrote {AB4.relative_to(ROOT)}/clusters_applied.gpkg ({sum(len(p) for p in parts)} polygons)")

core: tier 31 km2 -> 8 components, 1 >= 10 km2 (17 km2)
   min size  5 km2: 2 clusters, 23 km2 kept
   min size 10 km2: 1 clusters, 17 km2 kept
   min size 25 km2: 0 clusters, 0 km2 kept
   min size 50 km2: 0 clusters, 0 km2 kept
scenario s1@585   (Core-habitat-forward): 32 components, 5 kept after core subtraction (143 km2 residual)
scenario s1@245   (Core-habitat-forward): 33 components, 8 kept after core subtraction (706 km2 residual)
scenario s2       (Connectivity-forward): 40 components, 4 kept after core subtraction (82 km2 residual)
scenario s3@585   (Biodiversity-forward): 3 components, 0 kept after core subtraction (0 km2 residual)
scenario s3@245   (Biodiversity-forward): 2 components, 0 kept after core subtraction (0 km2 residual)
scenario s4@585   (Carbon-forward): 3 components, 0 kept after core subtraction (0 km2 residual)
scenario s4@245   (Carbon-forward): 3 components, 0 kept after core subtraction (0 km2 residual)
driver masks: m_soc theta-tail 716 disc cells | rare 

In [9]:
# ---- figures: F2 (guarded) + tiers + per-scenario small multiples ------------------------------------------------
rows_, cols_ = np.where(G.pu.any(axis=1))[0], np.where(G.pu.any(axis=0))[0]
r0, r1, c0, c1 = rows_.min() - 5, rows_.max() + 6, cols_.min() - 5, cols_.max() + 6
def show(ax, v1d, title, cmap="viridis", vmin=0, vmax=1, mask_locked=True):
    g2 = dc.to_grid(G, v1d.astype(np.float32))
    if mask_locked: g2[G.locked2d] = np.nan
    im = ax.imshow(g2[r0:r1, c0:c1], cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    lk = np.where(G.locked2d, 1.0, np.nan)[r0:r1, c0:c1]
    ax.imshow(lk, cmap=plt.matplotlib.colors.ListedColormap(["#bbbbbb"]), vmin=0, vmax=1, interpolation="nearest", alpha=0.6)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([]); return im
fig, ax = plt.subplots(1, 2, figsize=(11, 11))
show(ax[0], F2g, f"Ensemble F (applied g={APPLIED}%, guarded) -- additions only; PAs grey")
show(ax[1], tier_code.astype(np.float32) / 3.0, "tiers: 3 core / 2 value-forward / 1 opportunity / 0 never", cmap="YlOrRd")
fig.tight_layout(); fig.savefig(FIGS / "ab4_F_applied_tiers.png", dpi=170); plt.close(fig)
keys = [k for k in POOL if not (k.split("@")[0] in ("s0", "s5") or k.split("@")[0].endswith("x"))]
fig, axs = plt.subplots(1, max(len(keys), 1), figsize=(3.2 * max(len(keys), 1), 9))
for ax, k in zip(np.atleast_1d(axs), keys):
    show(ax, POOL[k], f"{dc.SCENARIO_LABEL.get(k.split('@')[0], k)} f (applied band)")
fig.tight_layout(); fig.savefig(FIGS / "ab4_scenario_f_applied.png", dpi=170); plt.close(fig)
print(f"wrote {FIGS.relative_to(ROOT)}/ab4_F_applied_tiers.png + ab4_scenario_f_applied.png")

wrote analyses/alberta_prioritization/figures/v3.1/ab4_F_applied_tiers.png + ab4_scenario_f_applied.png


In [10]:
# ---- summary record --------------------------------------------------------------------------------------------
SUMMARY = dict(created_utc=datetime.now(timezone.utc).isoformat(), version=VERSION, manifest=VP.manifest.name, efg_block=config.EFG_SUBDIR, n_efg=n_efg,
               efg_target_rule=(str(MAN.efg_target_rule.iloc[0]) if "efg_target_rule" in MAN.columns else "flat"),
               level="A", n_formulations=len(FORMS), threshold=THR, cluster_min_km2_provisional=MIN_KM2_AB, simplify_m=SIMPLIFY_M,
               applied_band_g=int(APPLIED) / 100, applied_band_rule=f"D-AB13: guarded 5% unless the 5% guarded frequent tier < {FLAT_KM2} km2, then 2% (D-AB10)",
               freq5_guarded_km2=freq5, flat_km2=FLAT_KM2, band_width_abs=BW.to_dict(orient="index"),
               bands_5pct=B5.to_dict(orient="records"), bands_applied=B2.to_dict(orient="records"),
               E1=dict(mean_abs=float(np.abs(bias[G.disc]).mean()), max_abs=float(np.abs(bias[G.disc]).max())),
               E3=E3, crossed_contrast=XREG, D_5=D["5_plain"], D_applied=D[f"{APPLIED}_plain"], E11=E11, C1=C1S,
               tiers_applied=ACT.to_dict(orient="records"), pooling=prep.to_dict(orient="records"),
               guard_vs_plain_tier_jaccard_applied=float(tierJ), C4=C4.to_dict(orient="records"),
               tenure_split=TEN.to_dict(orient="records"), n_clusters_kept=int(REG.kept.sum()))
(REC / "gate_ab4_summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"wrote {(REC / 'gate_ab4_summary.json').relative_to(ROOT)} (applied band g = {APPLIED}%) -- enter R8 in results_log.md; then 11b/11c (the necessity test) -> 12 -> 13")


wrote analyses/alberta_prioritization/spec/v3.1/gate_ab4_summary.json (applied band g = 2%) -- enter R8 in results_log.md; then 11b/11c (the necessity test) -> 12 -> 13
